# KG1 v73 ROBUSTO - kienngx replica (target 0.86) - Colab H100/A100

## Config kienngx EXATA (0.86 reproduzivel via OCTO CHECK):
- `target_modules="all-linear"` (PEFT resolve em save para list safe)
- LoRA `r=32, alpha=32, dropout=0.05`
- `lr=1e-4, bs=1, ga=4, epochs=2, max_length=2048`
- `optim="adamw_torch", cosine + warmup 0.1`
- Dataset: `train.csv OFICIAL Kaggle sample(n=1200, seed=42)`

## Robustness (v2):
- Cell 4: Unsloth com bypass stats check + **fallback automatico transformers direto** se Unsloth falhar
- Cell 5: LoRA compativel com ambos paths (Unsloth OU transformers+PEFT)
- Timeout HF 600s + HF_TRANSFER ativado

## Hardware recomendado:
- H100 80GB: ~2-3h treino (otimo)
- A100 40GB High-RAM: ~4-6h treino (ok)
- Fallback transformers: +2x lento (4-6h H100, 8-12h A100)

## Score expected: 0.86 +/- 0.01 (replica kienngx proven)

## Secrets obrigatorios (icone chave esquerda Colab):
- `HF_KEY` = seu HF token
- `KAGGLE_USERNAME` = felipe1983 (opcional, so para download train.csv)
- `KAGGLE_KEY` = seu Kaggle key (opcional)

In [ ]:
# Cell 1: GPU diagnostic + anti-idle
import os, torch, subprocess, sys
r = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv",
                   shell=True, capture_output=True, text=True)
print(r.stdout)
_lines = r.stdout.split(chr(10))
GPU_NAME = _lines[1].split(",")[0].strip() if len(_lines) > 1 else "UNKNOWN"
print(f"GPU detected: {GPU_NAME}")
print(f"Torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")

# Check VRAM >= 24GB
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM: {vram_gb:.1f}GB")
    if vram_gb < 24:
        print(f"!!! WARN: VRAM {vram_gb:.1f}GB < 24GB - pode dar OOM com Nemotron-30B NF4")
        print("    Recomendacao: Runtime -> Change runtime type -> A100 High-RAM ou H100")
    else:
        print(f"[OK] VRAM suficiente para Nemotron-30B NF4")

# Anti-idle JS
from IPython.display import display, Javascript
display(Javascript("function ClickConnect(){document.querySelector('colab-connect-button').click()};setInterval(ClickConnect, 60000)"))


In [ ]:
# Cell 2: Install Unsloth + deps
%%capture
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "trl>=0.16" "peft>=0.18.1" accelerate bitsandbytes
!pip install -q "transformers>=4.55" liger-kernel datasets
!pip install -q hf_transfer
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

# Verify versions
import trl, peft, transformers
assert trl.__version__ >= "0.16", f"TRL {trl.__version__} < 0.16"
print(f"TRL {trl.__version__} | PEFT {peft.__version__} | Transformers {transformers.__version__}")


In [ ]:
# Cell 3: Drive mount + HF secret
from google.colab import drive, userdata
drive.mount("/content/drive")

try:
    HF_TOKEN = userdata.get("HF_KEY")
except Exception:
    try:
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = os.environ.get("HF_TOKEN", "")
        print("WARN: configure HF_KEY ou HF_TOKEN no Colab Secrets!")

assert HF_TOKEN and HF_TOKEN.startswith("hf_"), f"Invalid HF_TOKEN: {HF_TOKEN[:10]}"
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

CKPT_DIR = "/content/drive/MyDrive/kg1_v73_unsloth_moe"
os.makedirs(CKPT_DIR, exist_ok=True)
print(f"Checkpoint dir: {CKPT_DIR}")
print(f"HF_TOKEN: {HF_TOKEN[:10]}... OK")


In [ ]:
# Cell 4: NF4 FORCADO via transformers direto (bypass Unsloth auto-decision)
# Objetivo: garantir NF4 15GB em vez de BF16 63GB. Cabe folgado em H100 80GB.
# 3 OOMs anteriores com BF16+MoE provaram que precisa NF4 para funcionar.
import os, torch
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "600"

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

MAX_SEQ = 2048
MODEL_ID = "unsloth/Nemotron-3-Nano-30B-A3B"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb,
    device_map="auto",
    token=HF_TOKEN,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
)

tok = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    trust_remote_code=True,
)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

USED_UNSLOTH = False
print(f"[OK] Model NF4 loaded via transformers direct")
print(f"Model class: {type(model).__name__}")
print(f"GPU mem used: {torch.cuda.memory_allocated()/1e9:.1f}GB")
# Esperado: ~15-18GB (vs 63GB BF16 que estourava com MoE)


In [ ]:
# Cell 5: LoRA kienngx baseline (SEM MoE BOMBA - garantido cabe)
# Decisao: MoE BOMBA estoura VRAM em H100 80GB. Voltamos ao kienngx proven 0.86.
# Para MoE precisaria GPU >= 120GB (A100 80GB x2 ou H200).
from peft import LoraConfig, get_peft_model, TaskType

TARGET_MODULES_NEMOTRON = [
    "in_proj", "out_proj",           # Mamba-2 (in_proj obrigatorio no gate)
    "q_proj", "k_proj", "v_proj", "o_proj",  # Attention
    "up_proj", "down_proj",          # MLP shared expert
]

cfg = LoraConfig(
    r=32,                            # kienngx: 32
    lora_alpha=32,                   # kienngx: 32 (ratio 1:1)
    lora_dropout=0.05,               # kienngx: 0.05 (volta ao original)
    bias="none",
    target_modules=TARGET_MODULES_NEMOTRON,
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
)

model = get_peft_model(model, cfg)
model.enable_input_require_grads()
print("[OK] LoRA applied via PEFT direct (kienngx baseline, sem MoE BOMBA)")
model.print_trainable_parameters()
# Esperado: ~80-100M trainable (kienngx replica, score target 0.86 proven)


In [ ]:
# Cell 6: Dataset train.csv OFICIAL Kaggle (kienngx replica: 1200 samples seed=42)
import pandas as pd
from datasets import Dataset

# Baixar train.csv oficial do Kaggle (Drive ou via API)
TRAIN_CSV = "/content/drive/MyDrive/kg1_train.csv"
if not os.path.exists(TRAIN_CSV):
    os.makedirs("/root/.kaggle", exist_ok=True)
    import shutil, json as _json
    kaggle_json_drive = "/content/drive/MyDrive/.kaggle/kaggle.json"
    if os.path.exists(kaggle_json_drive):
        shutil.copy(kaggle_json_drive, "/root/.kaggle/kaggle.json")
    else:
        try:
            kaggle_user = userdata.get("KAGGLE_USERNAME")
            kaggle_key = userdata.get("KAGGLE_KEY")
            with open("/root/.kaggle/kaggle.json", "w") as f:
                _json.dump({"username": kaggle_user, "key": kaggle_key}, f)
        except Exception as e:
            raise RuntimeError(f"Configure KAGGLE_USERNAME/KEY no Colab Secrets: {e}")
    os.chmod("/root/.kaggle/kaggle.json", 0o600)
    os.system("kaggle competitions download -c nvidia-nemotron-model-reasoning-challenge -f train.csv -p /content/")
    os.system("unzip -o /content/train.csv.zip -d /content/ 2>/dev/null || true")
    TRAIN_CSV = "/content/train.csv"

print(f"Loading train.csv from {TRAIN_CSV}")
df_full = pd.read_csv(TRAIN_CSV)
print(f"Train.csv full: {len(df_full)} rows | columns: {list(df_full.columns)}")

# kienngx EXATO: sample 1200 seed=42
SUBSAMPLE_SIZE = 1200
df = df_full.sample(n=SUBSAMPLE_SIZE, random_state=42).reset_index(drop=True)
print(f"Subsampled: {len(df)} rows")

PROMPT_COL = "prompt" if "prompt" in df.columns else "problem"
ANSWER_COL = "answer" if "answer" in df.columns else "solution"
assert PROMPT_COL in df.columns, f"No prompt column in {df.columns}"
assert ANSWER_COL in df.columns, f"No answer column in {df.columns}"

# Prompt template kienngx EXATO
PROMPT_SUFFIX = chr(10) + "Put your final answer inside \\boxed{}."

def format_kienngx(row):
    user_msg = str(row[PROMPT_COL]) + PROMPT_SUFFIX
    assistant_msg = str(row[ANSWER_COL])
    messages = [
        {"role": "user", "content": user_msg},
        {"role": "assistant", "content": assistant_msg},
    ]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

ds_train = Dataset.from_pandas(df).map(format_kienngx, num_proc=2, remove_columns=list(df.columns))
print(f"Formatted: {len(ds_train)} examples")
print(f"Sample text (first 400 chars):")
print(ds_train[0]["text"][:400])


In [ ]:
# Cell 7: SFT kienngx EXATA (max_length 2048, sem MoE cabe folgado)
from trl import SFTTrainer, SFTConfig
import threading, time, trl
print(f"Using TRL {trl.__version__}")

args = SFTConfig(
    output_dir=CKPT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    learning_rate=1e-4,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_steps=100,
    save_total_limit=3,
    bf16=True,
    optim="adamw_torch",
    max_length=2048,                # kienngx EXATO
    dataset_text_field="text",
    packing=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_grad_norm=1.0,
    report_to="none",
    push_to_hub=False,
    seed=42,
    remove_unused_columns=True,
    dataloader_num_workers=0,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=ds_train,
    args=args,
    processing_class=tok,
)

def monitor_mem():
    while True:
        try:
            m = torch.cuda.memory_allocated()/1e9
            p = torch.cuda.max_memory_allocated()/1e9
            print(f"[MEM] {m:.1f}GB / peak {p:.1f}GB")
        except: pass
        time.sleep(120)
threading.Thread(target=monitor_mem, daemon=True).start()

resume = None
if os.path.exists(CKPT_DIR):
    ckpts = [d for d in os.listdir(CKPT_DIR) if d.startswith("checkpoint-")]
    if ckpts:
        resume = True
        print("Resuming from existing checkpoint")

stats = trainer.train(resume_from_checkpoint=resume)
print(f"Training done: {stats}")
print(f"Final loss: {stats.training_loss:.4f} (esperado: 0.5-1.5 kienngx)")


In [ ]:
# Cell 8: Save + validate gate + gerar submission.zip + upload HF
import json, zipfile
from huggingface_hub import HfApi

FINAL_DIR = f"{CKPT_DIR}/final_adapter"
trainer.save_model(FINAL_DIR)
tok.save_pretrained(FINAL_DIR)
print(f"Final adapter saved: {FINAL_DIR}")
print(f"Files: {os.listdir(FINAL_DIR)}")

# === VALIDACAO submission gate (alinhado com kg1_submission_gate.py) ===
print(chr(10) + "=== VALIDACAO submission gate ===")
adapter_config_path = f"{FINAL_DIR}/adapter_config.json"
with open(adapter_config_path) as f:
    cfg = json.load(f)

target_modules = cfg.get("target_modules", [])
rank = cfg.get("r", cfg.get("lora_rank", 0))
print(f"target_modules: {target_modules}")
print(f"rank: {rank}")

errors = []
if not isinstance(target_modules, list) or not target_modules:
    errors.append("target_modules nao e list ou vazia")
else:
    if "in_proj" not in target_modules: errors.append("missing in_proj")
    if "gate_proj" in target_modules: errors.append("contains gate_proj (unconverted Mamba)")
    if "x_proj" in target_modules: errors.append("contains x_proj (unconverted Mamba)")
if rank > 32: errors.append(f"rank {rank} > 32 (vLLM limit)")

if errors:
    print(f"!!! GATE FAIL: {errors}")
else:
    print("[OK] adapter PASSA submission_gate local")

# === Gerar submission.zip PRONTO para Kaggle ===
SUBMISSION_ZIP = f"{CKPT_DIR}/submission.zip"
REQUIRED = ["adapter_config.json", "adapter_model.safetensors"]

with zipfile.ZipFile(SUBMISSION_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in REQUIRED:
        src = os.path.join(FINAL_DIR, f)
        if not os.path.exists(src):
            print(f"WARN: {f} nao encontrado")
            continue
        zf.write(src, arcname=f)
        print(f"  added to zip: {f} ({os.path.getsize(src)/1e6:.1f} MB)")

size_mb = os.path.getsize(SUBMISSION_ZIP)/1e6
print(f"submission.zip salvo: {SUBMISSION_ZIP} ({size_mb:.1f} MB)")

# === Upload to HF ===
api = HfApi(token=HF_TOKEN)
REPO_ID = "felipesp1983/kg1-nemotron-lora-v73-kienngx-replica"
api.create_repo(REPO_ID, private=True, exist_ok=True)
api.upload_folder(folder_path=FINAL_DIR, repo_id=REPO_ID, path_in_repo="final")
api.upload_file(path_or_fileobj=SUBMISSION_ZIP, repo_id=REPO_ID, path_in_repo="submission.zip")
print(f"Uploaded to HF: {REPO_ID}")

print(chr(10) + "=" * 60)
print("PROXIMOS PASSOS (Kaggle submit):")
print("=" * 60)
print(f"1. Download submission.zip de: {SUBMISSION_ZIP}")
print("2. No terminal local:")
print("   export KAGGLE_USERNAME=felipe1983 KAGGLE_KEY=<seu_key>")
print("   kaggle competitions submit -c nvidia-nemotron-model-reasoning-challenge \\\\")
print(f"       -f submission.zip -m \"v73 kienngx replica r=32 all-linear\"")
print("3. Aguardar score (~1-3h processamento Kaggle)")
print("4. Score esperado: 0.86 +/- 0.01")


## PIPELINE APOS V73 (se atingir 0.86)

### Roadmap 0.86 -> 0.88+:
1. **V73 UNSLOTH aqui** -> 0.86 baseline (este notebook)
2. **V74 CURRICULUM + TWO-STAGE SFT** (em `C:/tmp/kg1_v74_kaggle/`) -> 0.87 (+0.01)
   - Curriculum easy -> hard via huikang.dev data
   - Two-stage SFT (token-level -> sample-level norm)
3. **V74 DARE-TIES MERGE** (`KG1_v74_DARE_TIES_MERGE_COLAB.ipynb`) -> 0.87-0.88
4. **V74 INFERENCE KAGGLE** -> apenas validacao LOCAL (Kaggle usa script oficial)

### Pos 0.88:
- Multi-teacher distillation (DeepSeek-R1 + OpenMath-Nemotron-14B)
- Huikang bit manipulation strategy 85% (topic 690307)
- GenSelect train-time ranker
- Solver hybrid (SymPy equation + cipher decoder)

### Regras CRITICAS:
- **Kaggle submit limit: 5/dia** (00:00 UTC = 21:00 BRT reset)
- **NUNCA submit sem 99% certeza** score vai melhorar (regra do usuario)
- **Pre-score local obrigatorio** antes de queimar submit slot
- **Validation gate local** antes de upload: adapter_config.json DEVE ter in_proj, r<=32, sem gate_proj/x_proj

### Troubleshooting:
- Cell 4 Unsloth timeout -> fallback automatico transformers direto (+2x lento)
- Cell 4 OOM -> mude para GPU maior (A100 High-RAM ou H100)
- Cell 6 train.csv nao encontrado -> configure KAGGLE_USERNAME/KEY nos Secrets
- Cell 7 loss NaN -> pare training e investigue config
- Cell 8 GATE FAIL -> verifique target_modules + rank no adapter_config.json